In [8]:
import sys
from pathlib import Path

# so we can import load_data.py from src/
sys.path.append(str(Path.cwd().parents[1] / "02_python" / "src"))

import pandas as pd
from load_data import load_all_tables

pd.set_option("display.max_columns", None)

tables = load_all_tables()
for name, df in tables.items():
    print(f"{name:22s} {df.shape[0]:>8,} rows  x  {df.shape[1]} cols")

customers                99,441 rows  x  5 cols
orders                   99,441 rows  x  8 cols
order_items             112,650 rows  x  7 cols
payments                103,886 rows  x  5 cols
reviews                  99,224 rows  x  7 cols
products                 32,951 rows  x  9 cols
sellers                   3,095 rows  x  4 cols
geolocation            1,000,163 rows  x  5 cols
category_translation         71 rows  x  2 cols


## Profiling approach

For each table: row count, column types, null counts per column, and a duplicate
check on the primary key. The goal is to confirm and quantify what the Excel first
look surfaced, and catch anything Excel couldn't show — particularly null patterns
across every column at once, which a pivot table can't do in one view.

In [9]:
def profile_table(df: pd.DataFrame, name: str, key_col: str = None) -> dict:
    """
    Returns a summary dict for one table: row count, null counts per column,
    and duplicate count on the given key column, if provided.
    """
    summary = {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
    }

    null_counts = df.isnull().sum()
    nulls_present = null_counts[null_counts > 0]
    summary["columns_with_nulls"] = len(nulls_present)
    summary["null_detail"] = nulls_present.to_dict()

    if key_col and key_col in df.columns:
        summary["distinct_key_values"] = df[key_col].nunique()
        summary["duplicate_key_rows"] = df[key_col].duplicated().sum()

    return summary

In [10]:
# key column per table, where one exists — used for the duplicate check
key_cols = {
    "customers": "customer_id",
    "orders": "order_id",
    "order_items": None,          # grain is order_id + order_item_id together, not one column
    "payments": None,             # order_id repeats by design (installments/split payments)
    "reviews": "review_id",
    "products": "product_id",
    "sellers": "seller_id",
    "geolocation": None,          # not a unique-keyed dimension
    "category_translation": "product_category_name",
}

profiles = []
for name, df in tables.items():
    result = profile_table(df, name, key_col=key_cols.get(name))
    profiles.append(result)

profiling_summary = pd.DataFrame(profiles)
profiling_summary

,table,rows,columns,columns_with_nulls,null_detail,distinct_key_values,duplicate_key_rows
0,customers,99441,5,0,{},99441.0,0.0
1,orders,99441,8,3,"{'order_approved_at': 160, 'order_delivered_ca...",99441.0,0.0
2,order_items,112650,7,0,{},NaN,NaN
3,payments,103886,5,0,{},NaN,NaN
4,reviews,99224,7,2,"{'review_comment_title': 87656, 'review_commen...",98410.0,814.0
5,products,32951,9,8,"{'product_category_name': 610, 'product_name_l...",32951.0,0.0
6,sellers,3095,4,0,{},3095.0,0.0
7,geolocation,1000163,5,0,{},NaN,NaN
8,category_translation,71,2,0,{},71.0,0.0


In [11]:
for row in profiles:
    if row["columns_with_nulls"] > 0:
        print(f"\n{row['table']} — {row['columns_with_nulls']} column(s) with nulls:")
        for col, count in row["null_detail"].items():
            pct = count / row["rows"] * 100
            print(f"  {col:30s} {count:>7,} nulls  ({pct:.1f}%)")


orders — 3 column(s) with nulls:
  order_approved_at                  160 nulls  (0.2%)
  order_delivered_carrier_date     1,783 nulls  (1.8%)
  order_delivered_customer_date    2,965 nulls  (3.0%)

reviews — 2 column(s) with nulls:
  review_comment_title            87,656 nulls  (88.3%)
  review_comment_message          58,247 nulls  (58.7%)

products — 8 column(s) with nulls:
  product_category_name              610 nulls  (1.9%)
  product_name_lenght                610 nulls  (1.9%)
  product_description_lenght         610 nulls  (1.9%)
  product_photos_qty                 610 nulls  (1.9%)
  product_weight_g                     2 nulls  (0.0%)
  product_length_cm                    2 nulls  (0.0%)
  product_height_cm                    2 nulls  (0.0%)
  product_width_cm                     2 nulls  (0.0%)


In [13]:
# Are the products nulls concentrated in the same rows?
products_null_mask = tables["products"]["product_category_name"].isnull()
same_rows_check = tables["products"].loc[products_null_mask, 
    ["product_category_name", "product_name_lenght", "product_description_lenght", "product_photos_qty"]
].isnull().all(axis=1).sum()

print(f"Rows missing category_name: {products_null_mask.sum()}")
print(f"Of those, also missing all three other fields: {same_rows_check}")

# What's going on with duplicate review_id?
dup_reviews = tables["reviews"][tables["reviews"]["review_id"].duplicated(keep=False)]
print(f"\nRows involved in duplicate review_id: {len(dup_reviews)}")
dup_reviews.sort_values("review_id").head(6)

Rows missing category_name: 610
Of those, also missing all three other fields: 610

Rows involved in duplicate review_id: 1603


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53


In [14]:
profiling_summary.drop(columns=["null_detail"]).to_csv(
    "../outputs/profiling_summary.csv", index=False
)

## Profiling findings summary

- **Orders**: delivery pipeline nulls narrow as expected — 160 never approved →
  1,783 never handed to carrier → 2,965 never delivered. Consistent with the
  ~3% non-delivery rate seen in the Excel pass.
- **Products**: 610 rows (1.9%) are missing every descriptive field at once
  (category, name length, description length, photo count) — one underlying
  defect (incomplete listing), not four independent ones.
- **Reviews**: `review_id` is not a reliable primary key — 1,603 rows share a
  `review_id` with another row under a different `order_id`, with identical
  score, comment and timestamps. All review analysis in this project keys on
  `order_id` instead.
- **Reviews**: ~88% of rows have no comment title and ~59% have no comment
  message — expected, since the numeric score is mandatory and the comment
  is optional. Not treated as a defect.